# chromVAR Motif Enrichment Analysis

**Method:** chromVAR + modified limma workflow (best performer per benchmarking paper)  
**Reference:** https://pmc.ncbi.nlm.nih.gov/articles/PMC11534267/

## 1. Load Libraries

In [ ]:
library(DiffBind)             # to reload consensus peaks
library(chromVAR)             # motif deviation scoring
library(motifmatchr)          # match motifs to peaks
library(universalmotif)       # read JASPAR files + TFBSTools conversion
library(TFBSTools)            # PWMatrixList required by matchMotifs
library(jsonlite)             # parse HOCOMOCO annotation JSONL
library(BSgenome.Mmusculus.UCSC.mm10)  # genome sequence for GC bias
library(SummarizedExperiment)
library(GenomicRanges)
library(GenomeInfoDb)
library(BiocParallel)
library(limma)                # eBayes differential testing + quantile normalization
library(ggplot2)
library(ggrepel)              # non-overlapping labels on volcano/bubble plots
library(pheatmap)
library(dplyr)
library(tibble)
library(tidyr)                # pivot_longer for QC distributions plot

## 2. Configuration
Edit this cell before running. All paths and parameters are defined here.

In [ ]:
BAM_DIR <- "/coh_labs/mvandenbrink/users/pkaur/6_tff1/2_bulk_atac/atacseq_pipeline/0_broadpeak/results/bwa/merged_library"
OUT_DIR <- "/coh_labs/mvandenbrink/users/pkaur/6_tff1/2_bulk_atac/atacseq_pipeline/0_broadpeak/0_motif_enrichment/chromvar_motif_enrichment_results/"
DIFFBIND <-"/coh_labs/mvandenbrink/users/pkaur/6_tff1/2_bulk_atac/atacseq_pipeline/0_broadpeak/diffbind_results/"
FIGS <- "/coh_labs/mvandenbrink/users/pkaur/6_tff1/2_bulk_atac/atacseq_pipeline/0_broadpeak/0_motif_enrichment/chromvar_motif_enrichment_results/figures/"
HOCOMOCO_FILE <- "/coh_labs/mvandenbrink/users/pkaur/6_tff1/2_bulk_atac/atacseq_pipeline/0_broadpeak/0_motif_enrichment/chromvar_motif_enrichment_results/H14CORE_jaspar_format.txt"
HOCOMOCO_MOUSE_ANNOT <- "/coh_labs/mvandenbrink/users/pkaur/6_tff1/2_bulk_atac/atacseq_pipeline/0_broadpeak/0_motif_enrichment/chromvar_motif_enrichment_results/HOCOMOCO v14 Mouse Annotations.jsonl"


CONDITIONS   <- c("GFP", "TFF1")   # first = control, second = treatment
N_REPS       <- 3
CONTROL_COND <- "GFP"
TREAT_COND   <- "TFF1"

BAM_SUFFIX <- ".mLb.clN.sorted.bam"   # e.g. GFP_REP1.mLb.clN.sorted.bam

# --- chromVAR parameters ---
# Paper recommends 1000–2000. Using 1000 as the minimum acceptable value.
N_ITERATIONS <- 1000
PEAK_RESIZE  <- 300    # bp; standard chromVAR recommendation
N_CORES      <- 4      # parallel workers for computeDeviations


FDR_THRESH <- 0.05

dir.create(OUT_DIR, recursive = TRUE, showWarnings = FALSE)
cat("Output directory:", OUT_DIR, "\n")
cat("Background iterations:", N_ITERATIONS, "(paper recommends 1000-2000)\n")

## OPTIONAL: Load Pre-computed chromVAR Results

In [ ]:
cat("Loading pre-computed chromVAR results from", OUT_DIR, "\n")

dev     <- readRDS(file.path(OUT_DIR, "chromvar_deviations.rds"))
z_norm  <- readRDS(file.path(OUT_DIR, "chromvar_znorm.rds"))
results <- read.csv(file.path(OUT_DIR, "chromvar_motif_results.csv"),
                    stringsAsFactors = FALSE)

# Recreate direction factor used by figures
results$direction <- factor(results$direction,
                            levels = c(paste0(TREAT_COND, "_enriched"),
                                       paste0(CONTROL_COND, "_enriched"),
                                       "not_significant"))

cat("Loaded:\n",
    "  dev:    ", nrow(dev), "motifs x", ncol(dev), "samples\n",
    "  z_norm: ", nrow(z_norm), "motifs x", ncol(z_norm), "samples\n",
    "  results:", nrow(results), "motifs\n")
cat("Ready to run visualization sections (11+).\n")

## 3. Load Consensus Peaks from DiffBind Object

Reuse the DiffBind-defined 6,514-peak consensus set for consistency with the differential accessibility analysis.

In [ ]:
dba_obj  <- readRDS(file.path(DIFFBIND, "dba_object.rds"))
peaks_gr <- dba.peakset(dba_obj, bRetrieve = TRUE, DataType = DBA_DATA_GRANGES)

peaks_gr <- granges(peaks_gr)

peaks_gr <- keepStandardChromosomes(peaks_gr, pruning.mode = "coarse")

cat("Consensus peaks loaded:", length(peaks_gr), "\n")
print(head(peaks_gr))

## 4. Build SummarizedExperiment + Count Fragments


In [ ]:
sample_ids <- paste0(
    rep(CONDITIONS, each = N_REPS),
    "_REP",
    rep(seq_len(N_REPS), times = length(CONDITIONS))
)
bam_files <- file.path(BAM_DIR, paste0(sample_ids, BAM_SUFFIX))

# Verify BAM files exist
missing_bams <- bam_files[!file.exists(bam_files)]
if (length(missing_bams) > 0) {
    stop("Missing BAM files:\n", paste(missing_bams, collapse = "\n"))
}
cat("All", length(bam_files), "BAM files found.\n")

se <- getCounts(
    bam_files,
    peaks_gr,
    paired  = TRUE,
    by_rg   = FALSE,
    format  = "bam",
    colData = DataFrame(
        sample    = sample_ids,
        condition = factor(
            rep(CONDITIONS, each = N_REPS),
            levels = c(CONTROL_COND, TREAT_COND)
        )
    )
)

cat("SummarizedExperiment dimensions:", nrow(se), "peaks x", ncol(se), "samples\n")
cat("Count range:", range(assay(se)), "\n")

rowRanges(se) <- resize(rowRanges(se), width = PEAK_RESIZE, fix = "center")
cat("Peaks resized to", PEAK_RESIZE, "bp\n")

## 5. Add GC Bias and Filter Peaks

- `addGCBias()`: computes GC content per peak; used by `getBackgroundPeaks()` to select GC-matched background
- `filterPeaks()`: removes peaks with very low counts and overlapping peaks (standard chromVAR QC)

In [ ]:
se <- addGCBias(se, genome = BSgenome.Mmusculus.UCSC.mm10)
cat("GC bias added. GC content range:",
    round(range(rowData(se)$bias, na.rm = TRUE), 3), "\n")

counts_filtered <- filterPeaks(se, non_overlapping = TRUE)
cat("Peaks after filtering:", nrow(counts_filtered),
    "(removed", nrow(se) - nrow(counts_filtered), "peaks)\n")

## 6. Get HOCOMOCO Motifs from MotifDb

Per paper recommendation: use HOCOMOCO v10/v11 motifs.  
Strategy: keep 1 motif per gene symbol — prefer v11 over v10, then highest sequenceCount (most training sequences = best quality).

In [ ]:
for (f in c(HOCOMOCO_FILE, HOCOMOCO_MOUSE_ANNOT)) {
    if (!file.exists(f)) stop(
        "File not found: ", f,
        "\nSee the configuration cell for download instructions."
    )
}

cat("Parsing mouse annotation JSONL...\n")
annot_lines <- readLines(HOCOMOCO_MOUSE_ANNOT)
annot_list  <- lapply(annot_lines, jsonlite::fromJSON)

mouse_ids  <- sapply(annot_list, function(x) x$name)
mouse_qual <- sapply(annot_list, function(x) x$quality)
mouse_gsym <- sapply(annot_list, function(x) {
    gs <- tryCatch(
        x$masterlist_info$species$MOUSE$gene_symbol,
        error = function(e) NULL
    )
    if (is.null(gs) || length(gs) == 0 || is.na(gs) || gs == "") x$tf else gs
})

mouse_meta <- data.frame(
    name       = mouse_ids,
    geneSymbol = mouse_gsym,
    quality    = mouse_qual,
    stringsAsFactors = FALSE
)
cat("Mouse annotation entries:", nrow(mouse_meta), "\n")

cat("Reading JASPAR file...\n")
motif_all    <- read_jaspar(HOCOMOCO_FILE)
jaspar_names <- sapply(motif_all, function(m) m@name)
cat("Total JASPAR motifs loaded:", length(motif_all), "\n")

is_mouse    <- jaspar_names %in% mouse_meta$name
motif_mouse <- motif_all[is_mouse]
cat("Mouse motifs matched in JASPAR file:", length(motif_mouse),
    "(expected ~1245)\n")

if (length(motif_mouse) == 0) {
    stop("No mouse motifs matched — check that both files are from the same HOCOMOCO v14 release.")
}

matched_names <- jaspar_names[is_mouse]
meta_ordered  <- mouse_meta[match(matched_names, mouse_meta$name), ]

keep_idx <- meta_ordered %>%
    mutate(.idx = seq_len(n())) %>%
    arrange(geneSymbol, quality) %>%
    group_by(geneSymbol) %>%
    slice(1) %>%
    ungroup() %>%
    pull(.idx)

motif_obj  <- motif_mouse[keep_idx]
motif_meta <- meta_ordered[keep_idx, ]

cat("Unique gene symbols (1 motif each):", length(motif_obj),
    "(expected ~809)\n")
cat("Quality tier breakdown:\n")
print(table(motif_meta$quality))
cat("Example gene symbols:", paste(head(motif_meta$geneSymbol, 5), collapse = ", "), "\n")

motif_pwm_list <- convert_motifs(motif_obj, class = "TFBSTools-PWMatrix")
motif_pwm      <- do.call(TFBSTools::PWMatrixList, motif_pwm_list)
names(motif_pwm) <- motif_meta$geneSymbol   # e.g. "Foxa1", "Klf4"

cat("\nFinal PWMatrixList ready:", length(motif_pwm), "motifs\n")
cat("Class:", class(motif_pwm), "\n")
cat("Example names:", paste(head(names(motif_pwm), 5), collapse = ", "), "\n")

## 7. Match Motifs to Peaks

`matchMotifs()` scans each (300bp-resized) peak for each motif and returns a binary annotation matrix.

In [ ]:
motif_ix <- matchMotifs(
    motif_pwm,           
    counts_filtered,
    genome = BSgenome.Mmusculus.UCSC.mm10
)

cat("Motif match matrix dimensions:",
    nrow(motif_ix), "peaks x", ncol(motif_ix), "motifs\n")

match_rates <- colMeans(assay(motif_ix))
cat("Motif match rate (fraction of peaks containing motif):\n")
cat("  Median:", round(median(match_rates), 3), "\n")
cat("  Range: ", round(range(match_rates), 3), "\n")

## 8. Compute chromVAR Deviations

**Critical parameter:** `niterations = 1000`  
The default (500) produces irreproducible z-scores across different random seeds. The paper recommends 1000–2000 iterations minimum.

In [ ]:
# Register parallel backend
register(MulticoreParam(N_CORES))

cat("Computing background peaks with", N_ITERATIONS, "iterations...\n")
cat("(This corrects for GC content and peak accessibility bias)\n")

bg_peaks <- getBackgroundPeaks(
    object      = counts_filtered,
    niterations = N_ITERATIONS   # CRITICAL: 1000 minimum (default 500 insufficient)
)

cat("Background peaks computed. Computing deviations...\n")

dev <- computeDeviations(
    object           = counts_filtered,
    annotations      = motif_ix,
    background_peaks = bg_peaks
)

cat("Deviations computed. Dimensions:",
    nrow(dev), "motifs x", ncol(dev), "samples\n")
print(dev)

## 9. Extract Z-scores and Quantile Normalize


In [ ]:
z_mat <- deviationScores(dev)
cat("Z-score matrix:", nrow(z_mat), "motifs x", ncol(z_mat), "samples\n")

na_rows <- rowSums(is.na(z_mat)) == ncol(z_mat)
cat("Motifs with all-NA z-scores (will be excluded):", sum(na_rows), "\n")
z_mat <- z_mat[!na_rows, ]

# QC before normalization: column distributions should show per-sample shifts
cat("\nColumn means BEFORE quantile normalization:\n")
print(round(colMeans(z_mat, na.rm = TRUE), 3))

z_imputed <- z_mat
for (j in seq_len(ncol(z_imputed))) {
    col_mean <- mean(z_imputed[, j], na.rm = TRUE)
    z_imputed[is.na(z_imputed[, j]), j] <- col_mean
}
z_norm <- limma::normalizeBetweenArrays(z_imputed, method = "quantile")
rownames(z_norm) <- rownames(z_mat)
colnames(z_norm) <- colnames(z_mat)

z_norm[is.na(z_mat)] <- NA

cat("\nColumn means AFTER quantile normalization (should be ~equal):\n")
print(round(colMeans(z_norm, na.rm = TRUE), 3))

cat("\nQuantile normalization complete.\n")

## 10. Limma Differential Testing


In [ ]:
condition <- factor(
    colData(dev)$condition,
    levels = c(CONTROL_COND, TREAT_COND)
)

print(condition)
design <- model.matrix(~ condition)
colnames(design) <- c("Intercept", paste0(TREAT_COND, "_vs_", CONTROL_COND))

cat("Design matrix:\n")
print(design)


na_rows <- !complete.cases(z_norm)
if (any(na_rows)) {
    cat(sprintf("Excluding %d motifs with NA z-scores from limma (keeping for QC plots)\n",
                sum(na_rows)))
}
z_norm_fit <- z_norm[!na_rows, , drop = FALSE]

# Fit limma model and apply eBayes variance moderation
fit  <- lmFit(z_norm_fit, design)
fit  <- eBayes(fit)

# Extract all results sorted by p-value
coef_name <- paste0(TREAT_COND, "_vs_", CONTROL_COND)
results <- topTable(fit, coef = coef_name, number = Inf, sort.by = "P") %>%
    rownames_to_column("motif") %>%
    rename(
        log2FC     = logFC,
        avg_expr   = AveExpr,
        t_stat     = t,
        pval       = P.Value,
        padj       = adj.P.Val,
        B          = B
    ) %>%
    mutate(
        direction  = case_when(
            padj < FDR_THRESH & log2FC > 0 ~ TREAT_COND,
            padj < FDR_THRESH & log2FC < 0 ~ CONTROL_COND,
            TRUE                           ~ "NS"
        )
    )

cat("\nDifferential motif analysis results:\n")
cat(sprintf("  Total motifs tested: %d\n", nrow(results)))
cat(sprintf("  Significant (FDR < %.2f): %d\n", FDR_THRESH, sum(results$padj < FDR_THRESH)))
cat(sprintf("  %s-enriched: %d\n", TREAT_COND,   sum(results$direction == TREAT_COND)))
cat(sprintf("  %s-enriched: %d\n", CONTROL_COND, sum(results$direction == CONTROL_COND)))

cat("\nTop 10 motifs by p-value:\n")
print(head(results[, c("motif", "log2FC", "pval", "padj", "direction")], 10))

# Save results immediately — don't wait for the final Save Outputs cell.
# This ensures results are persisted even if downstream visualizations fail.
write.csv(results, file.path(OUT_DIR, "chromvar_motif_results.csv"), row.names = FALSE)
cat(sprintf("\nResults saved: chromvar_motif_results.csv (%d motifs)\n", nrow(results)))

## 11. Visualizations

Ranked Motif Deviation Plot

In [ ]:
HIGHLIGHT_MOTIFS <- c(
    "Pax1", "Pax2", "Pax3", "Pax4", "Pax5",
    "Pax6", "Pax7", "Pax8", "Pax9", 
    "Klf12","Atf3","Fos", "Fosb",'Junb',"Hoxd1","Hoxb1"
    # Add more motifs below, e.g.:
    # "EBF1", "EBF2", "SP1"
)

HIGHLIGHT_COLOR_POS <- "#E65100"   # orange — enriched in TREAT (log2FC >= 0)
HIGHLIGHT_COLOR_NEG <- "#2166AC"   # blue   — depleted in TREAT (log2FC < 0)

# Rank all motifs by log2FC ascending (left = depleted in TREAT, right = enriched)
ranked <- results %>%
    arrange(log2FC) %>%
    mutate(
        rank      = seq_len(n()),
        highlight = motif %in% HIGHLIGHT_MOTIFS
    )

bg_pts <- ranked %>% filter(!highlight)
hl_pts <- ranked %>% filter(highlight) %>%
    mutate(hl_color = ifelse(log2FC < 0, HIGHLIGHT_COLOR_NEG, HIGHLIGHT_COLOR_POS))

# Symmetric y-axis: equal distance above and below 0
y_lim <- max(abs(ranked$log2FC), na.rm = TRUE) * 1.05

# Report which requested motifs were found
found_motifs <- HIGHLIGHT_MOTIFS[HIGHLIGHT_MOTIFS %in% ranked$motif]
missing_motifs <- HIGHLIGHT_MOTIFS[!HIGHLIGHT_MOTIFS %in% ranked$motif]
if (length(found_motifs)   > 0) cat("Highlighted motifs found:  ", paste(found_motifs,   collapse = ", "), "\n")
if (length(missing_motifs) > 0) cat("Motifs not in results:     ", paste(missing_motifs, collapse = ", "), "\n")

p_ranked <- ggplot() +
    # Background points: black filled circles
    geom_point(
        data  = bg_pts,
        aes(x = rank, y = log2FC),
        color = "black", shape = 16, size = 2, alpha = 0.4
    ) +
    # Highlighted points: orange if log2FC >= 0, blue if log2FC < 0
    geom_point(
        data  = hl_pts,
        aes(x = rank, y = log2FC),
        color = hl_pts$hl_color, shape = 17, size = 5
    ) +
    # Labels for highlighted motifs — color matches the point
    geom_text_repel(
        data          = hl_pts,
        aes(x = rank, y = log2FC, label = motif),
        color              = hl_pts$hl_color,
        size               = 5,
        fontface           = "bold",
        max.overlaps       = 40,
        segment.color      = "grey40",
        segment.size       = 0.5,
        min.segment.length = 0,
        point.padding      = 1,
        box.padding        = 1,
        force              = 3
    ) +
    # Reference line at 0
    geom_hline(yintercept = 0, linetype = "dashed", color = "grey40") +
    # Symmetric y-axis around 0
    coord_cartesian(ylim = c(-y_lim, y_lim)) +
    labs(
        title    = paste("Ranked Motif Deviation Score Change:", TREAT_COND, "vs", CONTROL_COND),
        x = "Motif Rank",
        y = paste0("log2FC deviation score")
    ) +
    theme_bw(base_size = 12) +
    theme(
        plot.title  = element_text(face = "bold", hjust = 0.5),
        panel.grid  = element_blank()
    )

print(p_ranked)
ggsave(file.path(FIGS, "chromvar_ranked_motif_plot.pdf"), p_ranked)
cat("Saved: chromvar_ranked_motif_plot.pdf\n")

In [ ]:
write.csv(results,                                        file.path(OUT_DIR, "chromvar_motif_results.csv"),    row.names = FALSE)
write.csv(results %>% filter(padj < FDR_THRESH),          file.path(OUT_DIR, "chromvar_motif_sig.csv"),         row.names = FALSE)
saveRDS(dev,    file.path(OUT_DIR, "chromvar_deviations.rds"))
saveRDS(z_norm, file.path(OUT_DIR, "chromvar_znorm.rds"))

In [ ]:
sessionInfo()